# Faruq-v3 Breadth Screening Batch v1

This notebook runs the **frozen seed-42 validation-only breadth batch**. Candidate implementations are checked out by exact commit SHA from `configs/breadth_screening/faruq_v3_batch_v1.json`. Results are written to one persistent Drive ledger. The locked test split is never extracted or opened.

Run one chunk per Colab session if needed. Re-running the same chunk is safe: completed candidates are skipped and candidate runners may resume their own checkpoints. Candidate stdout/stderr is also persisted as `runner.log` under each candidate output root.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time, json
from pathlib import Path

# A previous run ends inside REPO. Move to a stable directory before
# deleting/re-cloning it, otherwise git inherits a deleted cwd.
os.chdir('/content')

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/breadth-screening-batch-v1'
REMOTE = 'https://github.com/ediprin/coffee-bean-detection.git'

if REPO.exists():
    shutil.rmtree(REPO)

clone = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REMOTE, str(REPO)],
    text=True, capture_output=True, cwd='/content',
)
if clone.returncode != 0:
    print(clone.stdout)
    print(clone.stderr)
    raise RuntimeError(f'git clone failed with exit code {clone.returncode}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True, cwd='/content')
# FTIF is the only common-batch candidate with this optional dependency.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'open_clip_torch'], check=True, cwd='/content')

sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
controller = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('controller:', controller)
print('branch:', BRANCH)
print('remote:', REMOTE)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan GPU runtime.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
D0FT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
GROUPED = DATA_ROOT/'faruq_grouped_summary.json'
assert GROUPED.is_file()
assert not (DATA_ROOT/'test').exists(), 'STOP: development archive unexpectedly exposes test.'
BATCH_ROOT = PROJECT_ROOT/'experiments/faruq-v3-breadth-screening-batch-v1'
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Drive batch root:', BATCH_ROOT)


In [ ]:
MANIFEST = json.loads((REPO/'configs/breadth_screening/faruq_v3_batch_v1.json').read_text())
enabled = [c for c in MANIFEST['candidates'] if c.get('enabled') and c.get('group') == 'detector']
print('Frozen protocol:', MANIFEST['protocol'])
print('Enabled common detector candidates:', len(enabled))
for i, c in enumerate(enabled, 1): print(f"{i:02d}. {c['id']:8s} {c['family']:28s} {c['sha'][:10]}")
print('Canonical gate:', MANIFEST['canonical_gate'])


## Choose one resumable chunk

Chunking is only for Colab runtime management. All chunks use the same frozen gate and write to the same result ledger.

In [ ]:
CHUNKS = {
    1: ['SG1','SSCB','MRL','APCL1','PCL1','CPE','BHCL','HIERVIP'],
    2: ['SAF1','DRNET','DRIV','SF1','CF1','SC1','STB1','IGEM','PWCA'],
    3: ['FBNR','SEMAUX','CG1','AFAB','FTIF'],
}
CHUNK_ID = 3  # resume/finalize the remaining breadth-screening candidates
ONLY = CHUNKS[CHUNK_ID]
print('Selected chunk', CHUNK_ID, ONLY)


In [ ]:
cmd = [
    sys.executable, '-u', str(REPO/'tools/run_faruq_v3_breadth_batch_logged.py'),
    '--repo', str(REPO),
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED),
    '--control-summary', str(CONTROL),
    '--d0-checkpoint', str(D0),
    '--d0ft-report', str(D0FT),
    '--batch-root', str(BATCH_ROOT),
    '--device', '0',
    '--continue-on-error',
    '--only', *ONLY,
]
print('RUN:', ' '.join(map(str, cmd)), flush=True)
process = subprocess.run(cmd, cwd=REPO)
print('batch controller exit code:', process.returncode)
if process.returncode != 0:
    print('Inspect master_state.json and candidate/runner.log. Completed candidates remain resumable; failures are not scientific results.')


In [ ]:
import pandas as pd
from IPython.display import display
CSV = BATCH_ROOT/'master_results.csv'
STATE = BATCH_ROOT/'master_state.json'
if CSV.is_file():
    frame = pd.read_csv(CSV)
    frame = frame.sort_values(['decision','delta_macro_vs_D0FT','delta_bottom3_vs_D0FT'], ascending=[False,False,False])
    display(frame)
    print('RETAIN rows:', int((frame['decision'] == 'RETAIN').sum()), '/', len(frame))
else:
    print('No master_results.csv yet.')
if STATE.is_file():
    state = json.loads(STATE.read_text())
    print({key: value.get('status') for key, value in state.get('candidates', {}).items()})
print('CSV:', CSV)
print('STATE:', STATE)


## After all three chunks

Do **not** touch the locked test. Use `master_results.csv` to choose approximately 5–8 mechanistically distinct survivors. Only then define the combination phase and paired multi-seed confirmation. DC² diagnostics remain a separate track and are not mixed into this common detector gate.